# **6. Wiring the forecast into the app**

Chapters 2 to 5 produced a validated forecaster and an advisor. Chapter 6 makes them part of
Loca-T: a service the other layers can call, an endpoint for the dashboard, a USSD menu and
an SMS digest for hosts. The app code lives in `src/locat/`; this notebook records each step
and checks it by calling the running server.

## **6.1 The advice service and its endpoint**

### **6.1.1 Why**
The forecaster and advisor are modules, but nothing in the app calls them. The first piece
is a service that fits the forecaster once, forecasts for a property starting on demo day,
and runs the advisor. A JSON endpoint exposes it, so the dashboard and later steps use one
path.

### **6.1.2 What gets built**
1. `advice_service.py`: loads the saved data and fits the forecaster once (cached). Forecasts
   start today, or on 26 November if today is earlier (the day after the demo history ends),
   and never beyond 60 days or the end of the calendar (31 March 2027).
2. `GET /advice/{listing_code}?days=N`: night-by-night forecast, range, level, action, price and
   reasons, labelled as synthetic data with the validated horizon.
3. A setting `DEMO_TODAY` to fix the demo date for rehearsals (empty means today).
4. Three tests on the service.

### **6.1.3 Predictions (written before the run)**
1. `pytest` reports 8 passed (5 for the advisor, 3 new).
2. `GET /advice/XESD2C?days=28` returns 200 and 28 nights, starting on 2026-11-26 (while today
   is on or before that date), with `synthetic_data` true.
3. With `days=60`: 60 nights, the last on 2027-01-24, and actions of hold 41, promote 15 and
   test 4, with 14 busy nights, exactly as in 5.11.
4. The festive rows (25 and 26 December, 1 and 2 January) are test at KES 4,700, with ranges
   of 85-98% and 83-98%.
5. An unknown listing code returns 404, and `days=500` returns 60 nights, not 500.

### **6.1.4 Decision rule**
If all five hold, the service is accepted and step 6.3 adds the USSD outlook and the SMS
digest. If a prediction fails, we find out why first.

In [1]:
import httpx
import pandas as pd

BASE = "http://127.0.0.1:8020"
CODE = "XESD2C"  # Bibi Akinyi Homestay's listing code from Phase 4a

r28 = httpx.get(f"{BASE}/advice/{CODE}", params={"days": 28}, timeout=120)
r60 = httpx.get(f"{BASE}/advice/{CODE}", params={"days": 60}, timeout=120)
print("status:", r28.status_code, r60.status_code)
d60 = r60.json()
print({k: d60[k] for k in ["listing_code", "property", "destination", "base_price",
                           "synthetic_data", "validated_horizon_days"]})

n28, n60 = pd.DataFrame(r28.json()["nights"]), pd.DataFrame(d60["nights"])
print(f"\nnights: {len(n28)} and {len(n60)} | first night {n60['date'].iloc[0]} | last night {n60['date'].iloc[-1]}")
print("actions (60 days):", n60["action"].value_counts().to_dict(),
      "| busy nights:", int((n60["level"] == "busy").sum()))

festive = n60[n60["date"].between("2026-12-24", "2027-01-03")]
print("\nFestive nights:")
print(festive[["day", "forecast_pct", "lower_pct", "upper_pct", "level",
               "action", "suggested_price", "reasons"]].to_string(index=False))

print("\nunknown code:", httpx.get(f"{BASE}/advice/ZZZZZZ").status_code)
big = httpx.get(f"{BASE}/advice/{CODE}", params={"days": 500}, timeout=120).json()
print("days=500 returns", len(big["nights"]), "nights")

status: 200 200
{'listing_code': 'XESD2C', 'property': 'Bibi Akinyi Homestay', 'destination': 'Naivasha', 'base_price': 4500.0, 'synthetic_data': True, 'validated_horizon_days': 60}

nights: 28 and 60 | first night 2026-11-26 | last night 2027-01-24
actions (60 days): {'hold': 41, 'promote': 15, 'test': 4} | busy nights: 14

Festive nights:
       day  forecast_pct  lower_pct  upper_pct  level action  suggested_price                                             reasons
Thu 24 Dec            44         38         51 normal   hold             4500                                     school holidays
Fri 25 Dec            97         85         98   busy   test             4700 school holidays, long weekend, weekend, payday week
Sat 26 Dec            96         85         98   busy   test             4700 school holidays, long weekend, weekend, payday week
Sun 27 Dec            63         55         74   busy   hold             4500          school holidays, long weekend, payday week
Mon 28 